In [ ]:
# ============================================================
# Comprehensive D2B QC Evaluation Script (N1–N4, 12 Plates)
# Author: Yu-Ting Kao | Final version (Nov 2025, viridis update)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import re

# ----------------------------
# 1. USER SETTINGS
# ----------------------------
data_folder = Path("/Users/gaoyuting/Desktop/1536 design/05. data analysis/Round 3/cell viability/QC files")   # e.g., Path("data/QC_plates")
pattern = "QC_plate*.csv"

# ----------------------------
# 2. FUNCTIONS
# ----------------------------
def robust_cv(series):
    """Robust coefficient of variation (%) using MAD."""
    median = np.median(series)
    mad = 1.4826 * np.median(np.abs(series - median))
    return 100 * mad / median if median != 0 else np.nan

def qc_block(name, nc, dmso, blank):
    """Compute QC metrics for one scaffold block."""
    nc_mean, nc_std = nc.mean(), nc.std()
    nc_cv = 100 * nc_std / nc_mean
    nc_rcv = robust_cv(nc)
    nc_median = np.median(nc)
    nc_mad = 1.4826 * np.median(np.abs(nc - nc_median))
    upper, lower = nc_median + 3 * nc_mad, nc_median - 3 * nc_mad
    false_alarm = np.mean((nc > upper) | (nc < lower)) * 100
    delta_dmso = dmso.mean() - nc_mean
    delta_blank = blank.mean() - nc_mean if len(blank) else np.nan
    return {
        "Block": name,
        "NC_mean": round(nc_mean, 4),
        "NC_%CV": round(nc_cv, 2),
        "NC_rCV": round(nc_rcv, 2),
        "NC_median": round(nc_median, 4),
        "DMSO_mean": round(dmso.mean(), 4),
        "Δ(DMSO–NC)": round(delta_dmso, 4),
        "Δ(Blank–NC)": round(delta_blank, 4),
        "False_alarm_%": round(false_alarm, 2),
    }

def plate_qc(filepath):
    """Evaluate both scaffold blocks for one plate."""
    df = pd.read_csv(filepath)
    df.columns = ["well", "viability"]
    df["col"] = df["well"].str.extract(r"(\d+)$").astype(int)

    plate_id = int(re.findall(r"\d+", filepath.stem)[0])  # e.g., 1–12
    if plate_id <= 6:
        block1_name, block2_name = "N1", "N2"
    else:
        block1_name, block2_name = "N3", "N4"

    nc1 = df[df["col"] == 11]["viability"]
    nc2 = df[df["col"] == 23]["viability"]
    dmso = df[df["col"] == 12]["viability"]
    blank = df[df["col"] == 24]["viability"]

    out = []
    if len(nc1) and len(dmso):
        b1 = qc_block(block1_name, nc1, dmso, blank)
        b1["Plate"] = filepath.stem
        out.append(b1)
    if len(nc2) and len(dmso):
        b2 = qc_block(block2_name, nc2, dmso, blank)
        b2["Plate"] = filepath.stem
        out.append(b2)
    return out

# ----------------------------
# 3. PROCESS ALL 12 PLATES (NUMERIC ORDER)
# ----------------------------
def plate_num_from_name(path_obj):
    m = re.search(r"(\d+)", path_obj.stem)
    return int(m.group(1)) if m else 0

all_results = []
files = sorted(data_folder.glob(pattern), key=plate_num_from_name)
for f in files:
    all_results.extend(plate_qc(f))

qc_df = pd.DataFrame(all_results)

# --- Add numeric plate order for plotting ---
qc_df["PlateNum"] = qc_df["Plate"].str.extract(r"(\d+)").astype(int)
qc_df = qc_df.sort_values("PlateNum")
ordered_plate_labels = [
    p for p, _ in sorted(
        qc_df[["Plate", "PlateNum"]].drop_duplicates().values,
        key=lambda x: int(re.search(r"\d+", x[0]).group())
    )
]
qc_df["Plate"] = pd.Categorical(qc_df["Plate"], categories=ordered_plate_labels, ordered=True)

# ----------------------------
# 4. SUMMARIES
# ----------------------------
plate_summary = (
    qc_df.groupby("Plate")
    .agg({
        "NC_rCV": "mean",
        "NC_%CV": "mean",
        "False_alarm_%": "mean",
        "Δ(DMSO–NC)": "mean",
        "Δ(Blank–NC)": "mean",
    })
    .rename(columns={
        "NC_rCV": "Mean_rCV",
        "NC_%CV": "Mean_%CV",
        "False_alarm_%": "Mean_FalseAlarm",
        "Δ(DMSO–NC)": "Mean_Δ_DMSONC",
        "Δ(Blank–NC)": "Mean_Δ_BlankNC",
    })
    .reset_index()
)

overall_summary = {
    "Mean_rCV": qc_df["NC_rCV"].mean(),
    "SD_rCV": qc_df["NC_rCV"].std(),
    "Mean_%CV": qc_df["NC_%CV"].mean(),
    "SD_%CV": qc_df["NC_%CV"].std(),
    "Mean_FalseAlarm": qc_df["False_alarm_%"].mean(),
}

print("=== Per-block QC metrics ===")
print(qc_df)
print("\n=== Per-plate averaged QC ===")
print(plate_summary)
print("\n=== Overall assay summary ===")
for k, v in overall_summary.items():
    print(f"{k}: {v:.2f}")

# ----------------------------
# 5. SAVE QC DATA + FIGURES
# ----------------------------
output_folder = data_folder

qc_blocks_path = output_folder / "D2B_QC_blocks_N1-N4.csv"
plate_summary_path = output_folder / "D2B_QC_per_plate_N1-N4.csv"

qc_df.to_csv(qc_blocks_path, index=False)
plate_summary.to_csv(plate_summary_path, index=False)

print(f"\n✅ QC block data saved to: {qc_blocks_path}")
print(f"✅ Per-plate summary saved to: {plate_summary_path}")

mean_rCV = qc_df["NC_rCV"].mean()
sd_rCV = qc_df["NC_rCV"].std()
title_stats = f"(Avg rCV = {mean_rCV:.2f}% ± {sd_rCV:.2f}%)"

plt.rcParams['font.family'] = 'Arial'

# --- Levey–Jennings Chart (numeric order, custom plasma-inspired palette) ---
fig1, ax1 = plt.subplots(figsize=(9, 4))
x_labels = ordered_plate_labels
x_pos = np.arange(len(x_labels))

# Define consistent N1–N4 colors
color_map = {
    "N1": "#f4e254",
    "N2": "#d97461",
    "N3": "#a83d88",
    "N4": "#52299F"
}

for block in sorted(qc_df["Block"].unique()):
    sub = qc_df[qc_df["Block"] == block].sort_values("PlateNum")
    block_indices = [x_labels.index(p) for p in sub["Plate"]]
    ax1.plot(
        block_indices,
        sub["NC_median"].values,
        marker="o",
        markersize=6,
        linewidth=2,
        label=block,
        color=color_map.get(block, "#000000"),
    )

ax1.set_xticks(x_pos)
ax1.set_xticklabels(x_labels, rotation=45, ha="right")
ax1.set_ylabel("NC Median Viability")
ax1.set_title(f"Levey–Jennings Chart of NC Medians (N1–N4) {title_stats}")
ax1.legend(title="Scaffold Block", frameon=False, loc="best")
ax1.grid(alpha=0.3)
fig1.tight_layout()

fig1_path = output_folder / "Levey_Jennings_N1-N4.png"
fig1.savefig(fig1_path, dpi=300)
plt.close(fig1)
print(f"📊 Levey–Jennings figure saved to: {fig1_path}")

# --- rCV Distribution (4 split panels, simple y-axis: 0–2) ---
fig2, axes = plt.subplots(1, 4, figsize=(12, 4), sharey=True)
plt.rcParams["font.family"] = "Arial"

# Consistent custom color palette
color_map = {
    "N1": "#f4e254",  # yellow
    "N2": "#d97461",  # coral
    "N3": "#a83d88",  # magenta
    "N4": "#52299F"   # violet
}

bins = np.linspace(0, 30, 10)  # consistent bins for comparison

for ax, block in zip(axes, sorted(qc_df["Block"].unique())):
    subset = qc_df[qc_df["Block"] == block]
    color = color_map.get(block, "#000000")
    ax.hist(
        subset["NC_rCV"],
        bins=bins,
        color=color,
        edgecolor="black",
        linewidth=0.7,
        alpha=0.85
    )
    ax.set_title(block, fontsize=11, fontweight="bold")
    ax.set_xlabel("Robust %CV")
    ax.set_xlim(0, 30)
    ax.set_ylim(0, 2.2)                # fix y-axis limit
    ax.set_yticks([0, 1, 2])           # simple ticks
    ax.grid(alpha=0.25)
    if ax == axes[0]:
        ax.set_ylabel("Count")

fig2.suptitle(
    f"Distribution of Robust %CV by Scaffold Block {title_stats}",
    fontsize=12, fontweight="bold", y=1.03
)

fig2.tight_layout()
fig2_path = output_folder / "rCV_distribution_N1-N4_split_simpleY.png"
fig2.savefig(fig2_path, dpi=300, bbox_inches="tight")
plt.close(fig2)
print(f"📈 Split rCV distribution figure saved to: {fig2_path}")


# ----------------------------
# 6. HEATMAP (numeric order, viridis colormap)
# ----------------------------
# --- Heatmap (numeric order, plasma-inspired palette for consistency) ---
heat_df = qc_df.pivot_table(index="Plate", columns="Block", values="NC_rCV", aggfunc="mean")
heat_df = heat_df.reindex(index=ordered_plate_labels)

fig3, ax3 = plt.subplots(figsize=(7, 5))
im = ax3.imshow(heat_df.values, cmap="plasma", aspect="auto", vmin=0, vmax=20)

# Annotate cells
for i in range(heat_df.shape[0]):
    for j in range(heat_df.shape[1]):
        val = heat_df.values[i, j]
        if pd.notna(val):
            ax3.text(j, i, f"{val:.1f}", ha="center", va="center", color="white", fontsize=9)

ax3.set_xticks(range(len(heat_df.columns)))
ax3.set_xticklabels(heat_df.columns)
ax3.set_yticks(range(len(heat_df.index)))
ax3.set_yticklabels(heat_df.index)
ax3.set_xlabel("Scaffold Block")
ax3.set_ylabel("Plate")
ax3.set_title(f"D2B rCV Heatmap per Plate and Scaffold\n(Avg rCV = {mean_rCV:.2f}% ± {sd_rCV:.2f}%)")

cbar = fig3.colorbar(im, ax=ax3)
cbar.set_label("Robust %CV")

fig3.tight_layout()
fig3_path = output_folder / "D2B_rCV_heatmap_N1-N4.png"
fig3.savefig(fig3_path, dpi=300)
plt.close(fig3)
print(f"🧭 Heatmap saved to: {fig3_path}")
